# 🛒 Superstore Sales — Tam Analiz & Business Insights

**Dataset:** Global Superstore (9,800 sətir, 18 sütun)  
**Analitik:** Rahima | Devlab Data Internship  
**Məqsəd:** Satış performansını anlamaq, müştəri seqmentlərini müqayisə etmək, regionlar və kateqoriyalar üzrə tendensiyaları müəyyən etmək

---

## 1. Kitabxanalar və Data Yükləmə

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Stil tənzimləmələri
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'
PALETTE = ['#2D6A4F', '#40916C', '#52B788', '#74C69D', '#D8F3DC',
           '#1B4332', '#95D5B2', '#B7E4C7', '#F4A261', '#E76F51']
sns.set_style('whitegrid')

df = pd.read_csv('train.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Data Təmizliyi & Ön Emal

In [ ]:
# Data tipləri
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  dayfirst=True)

# Çatdırılma müddəti (gün)
df['Delivery Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Zaman əlavələri
df['Year']    = df['Order Date'].dt.year
df['Month']   = df['Order Date'].dt.month
df['Quarter'] = df['Order Date'].dt.quarter
df['YearMonth'] = df['Order Date'].dt.to_period('M')

# Postal Code boşluqlarını doldur
df['Postal Code'] = df['Postal Code'].fillna(0).astype(int)

# Boşluq yoxlaması
print('Null dəyərlər:')
print(df.isnull().sum()[df.isnull().sum() > 0])

print(f'\nÜmumi sifariş sayı: {df["Order ID"].nunique()}')
print(f'Unikal müştəri:      {df["Customer ID"].nunique()}')
print(f'Tarix aralığı:       {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'Ümumi satış:        ${df["Sales"].sum():,.0f}')

In [ ]:
# Əsas statistika
df[['Sales', 'Delivery Days']].describe().round(2)

## 3. Satış Tendensiyaları — Zaman Üzrə

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('📈 Satış Tendensiyaları', fontsize=16, fontweight='bold', y=1.01)

# 1. Aylıq satış trendi
ax1 = axes[0, 0]
monthly = df.groupby('YearMonth')['Sales'].sum().reset_index()
monthly['YearMonth_str'] = monthly['YearMonth'].astype(str)
ax1.plot(monthly['YearMonth_str'], monthly['Sales'], color='#2D6A4F', linewidth=2.2, marker='o', markersize=3)
ax1.fill_between(monthly['YearMonth_str'], monthly['Sales'], alpha=0.15, color='#2D6A4F')
ax1.set_title('Aylıq Satış Trendi', fontweight='bold')
ax1.set_xlabel('')
ax1.set_ylabel('Satış ($)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
tick_step = max(1, len(monthly) // 8)
ax1.set_xticks(range(0, len(monthly), tick_step))
ax1.set_xticklabels(monthly['YearMonth_str'][::tick_step], rotation=45, ha='right', fontsize=8)

# 2. İllik satış müqayisəsi
ax2 = axes[0, 1]
yearly = df.groupby('Year')['Sales'].sum()
bars = ax2.bar(yearly.index.astype(str), yearly.values, color=PALETTE[:len(yearly)], edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, yearly.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1500,
             f'${val:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.set_title('İllik Satış Həcmi', fontweight='bold')
ax2.set_ylabel('Satış ($)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# 3. Rüblük satış (heatmap şəklində bar)
ax3 = axes[1, 0]
quarterly = df.groupby(['Year', 'Quarter'])['Sales'].sum().unstack('Quarter')
quarterly.plot(kind='bar', ax=ax3, colormap='Greens', edgecolor='white')
ax3.set_title('Rüblük Satış (İllər üzrə)', fontweight='bold')
ax3.set_xlabel('İl')
ax3.set_ylabel('Satış ($)')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax3.legend(title='Rüb', labels=['Q1','Q2','Q3','Q4'])
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)

# 4. Ay üzrə ortalama satış (mövsümiyyət)
ax4 = axes[1, 1]
month_avg = df.groupby('Month')['Sales'].mean()
month_names = ['Yan','Fev','Mar','Apr','May','İyn','İyl','Avq','Sen','Okt','Noy','Dek']
bars4 = ax4.bar(range(1, 13), month_avg.values,
                color=['#E76F51' if v == month_avg.max() else '#52B788' for v in month_avg.values],
                edgecolor='white')
ax4.set_xticks(range(1, 13))
ax4.set_xticklabels(month_names, fontsize=9)
ax4.set_title('Mövsümiyyət — Aylıq Ortalama Satış', fontweight='bold')
ax4.set_ylabel('Ortalama Satış ($)')
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('01_time_trends.png', bbox_inches='tight', dpi=130)
plt.show()

# Insight
best_q = quarterly.sum().idxmax()
print(f'\n💡 Insight: Ən güclü rüb Q{best_q} — ilin sonuna doğru satışlar artır (mövsümsel pik).')

## 4. Kateqoriya & Alt-Kateqoriya Analizi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('📦 Kateqoriya Performansı', fontsize=15, fontweight='bold')

# 1. Kateqoriya üzrə ümumi satış (pie)
ax1 = axes[0]
cat_sales = df.groupby('Category')['Sales'].sum()
colors_pie = ['#2D6A4F', '#52B788', '#F4A261']
wedges, texts, autotexts = ax1.pie(
    cat_sales.values, labels=cat_sales.index,
    autopct='%1.1f%%', startangle=140,
    colors=colors_pie, pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=2))
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight('bold')
ax1.set_title('Kateqoriya Payı (Satış $)', fontweight='bold')

# 2. Alt-kateqoriya satış sıralaması
ax2 = axes[1]
sub_sales = df.groupby('Sub-Category')['Sales'].sum().sort_values(ascending=True)
colors_bar = ['#E76F51' if v == sub_sales.max() else '#52B788' for v in sub_sales.values]
bars = ax2.barh(sub_sales.index, sub_sales.values, color=colors_bar, edgecolor='white')
ax2.set_title('Alt-Kateqoriya Satış Sıralaması', fontweight='bold')
ax2.set_xlabel('Satış ($)')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
for bar, val in zip(bars, sub_sales.values):
    ax2.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
             f'${val:,.0f}', va='center', fontsize=7.5)

# 3. Kateqoriya üzrə sifariş sayı vs ortalama satış
ax3 = axes[2]
cat_agg = df.groupby('Category').agg(order_count=('Order ID','count'), avg_sale=('Sales','mean')).reset_index()
scatter = ax3.scatter(cat_agg['order_count'], cat_agg['avg_sale'],
                      s=[cat_sales[c]/80 for c in cat_agg['Category']],
                      c=colors_pie, alpha=0.85, edgecolors='white', linewidth=1.5)
for _, row in cat_agg.iterrows():
    ax3.annotate(row['Category'], (row['order_count'], row['avg_sale']),
                 textcoords='offset points', xytext=(8, 3), fontsize=9, fontweight='bold')
ax3.set_xlabel('Sifariş Sayı')
ax3.set_ylabel('Ortalama Satış ($)')
ax3.set_title('Həcm vs Ortalama Satış\n(nöqtə ölçüsü = ümumi satış)', fontweight='bold')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('02_category_analysis.png', bbox_inches='tight', dpi=130)
plt.show()

print('💡 Insight: Technology az sifarişlə ən yüksək ortalama satışı verir → premium məhsullar.')
print('💡 Insight: Office Supplies çox sifariş alsada, ortalama satışı aşağıdır → kəmiyyət bazlı.')

## 5. Region & Seqment Analizi

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('🌍 Region & Müştəri Seqmenti', fontsize=15, fontweight='bold')

# 1. Region üzrə satış
ax1 = axes[0, 0]
reg_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
bars = ax1.bar(reg_sales.index, reg_sales.values,
               color=['#2D6A4F','#40916C','#52B788','#74C69D'], edgecolor='white', linewidth=1)
for bar, val in zip(bars, reg_sales.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
             f'${val:,.0f}', ha='center', fontsize=9, fontweight='bold')
ax1.set_title('Region üzrə Ümumi Satış', fontweight='bold')
ax1.set_ylabel('Satış ($)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# 2. Seqment üzrə satış
ax2 = axes[0, 1]
seg_sales = df.groupby('Segment')['Sales'].sum().sort_values(ascending=False)
colors_seg = ['#F4A261', '#E76F51', '#2D6A4F']
wedges, texts, autotexts = ax2.pie(
    seg_sales.values, labels=seg_sales.index,
    autopct='%1.1f%%', colors=colors_seg,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    startangle=90)
for at in autotexts:
    at.set_fontsize(10); at.set_fontweight('bold')
ax2.set_title('Seqment Payı (Satış $)', fontweight='bold')

# 3. Region x Seqment heatmap
ax3 = axes[1, 0]
pivot_rs = df.pivot_table(values='Sales', index='Region', columns='Segment', aggfunc='sum')
sns.heatmap(pivot_rs, ax=ax3, annot=True, fmt=',.0f',
            cmap='Greens', linewidths=0.5, linecolor='white',
            annot_kws={'size': 9, 'weight': 'bold'})
ax3.set_title('Region × Seqment Satış Matrisi ($)', fontweight='bold')
ax3.set_xlabel(''); ax3.set_ylabel('')

# 4. Ship Mode üzrə sifariş paylaşımı
ax4 = axes[1, 1]
ship_counts = df['Ship Mode'].value_counts()
bars4 = ax4.bar(ship_counts.index, ship_counts.values,
                color=['#2D6A4F','#52B788','#F4A261','#E76F51'],
                edgecolor='white')
for bar, val in zip(bars4, ship_counts.values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=9, fontweight='bold')
ax4.set_title('Çatdırılma Üsulu (Sifariş Sayı)', fontweight='bold')
ax4.set_ylabel('Sifariş Sayı')
ax4.set_xticklabels(ship_counts.index, rotation=15, ha='right')

plt.tight_layout()
plt.savefig('03_region_segment.png', bbox_inches='tight', dpi=130)
plt.show()

top_reg = reg_sales.idxmax()
print(f'💡 Insight: {top_reg} regionu ən yüksək satışı gətirir.')
print('💡 Insight: Standard Class sifarişlərin yarısından çoxunu təşkil edir → sürətli çatdırılmaya tələb aşağıdır.')

## 6. Top Müştərilər & Çatdırılma Analizi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('👥 Müştəri & Çatdırılma Analizi', fontsize=15, fontweight='bold')

# 1. Top 10 müştəri
ax1 = axes[0]
top_customers = df.groupby('Customer Name')['Sales'].sum().sort_values(ascending=False).head(10)
colors_top = ['#E76F51' if i == 0 else '#52B788' for i in range(10)]
bars = ax1.barh(top_customers.index[::-1], top_customers.values[::-1],
                color=colors_top[::-1], edgecolor='white')
for bar, val in zip(bars, top_customers.values[::-1]):
    ax1.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
             f'${val:,.0f}', va='center', fontsize=8)
ax1.set_title('Top 10 Müştəri (Satış)', fontweight='bold')
ax1.set_xlabel('Ümumi Satış ($)')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# 2. Çatdırılma müddəti boxplot
ax2 = axes[1]
ship_del = df.groupby('Ship Mode')['Delivery Days'].apply(list).reset_index()
ship_order = ['Same Day', 'First Class', 'Second Class', 'Standard Class']
data_box = [df[df['Ship Mode'] == sm]['Delivery Days'].dropna().values for sm in ship_order]
bp = ax2.boxplot(data_box, patch_artist=True, notch=False,
                 medianprops={'color': 'white', 'linewidth': 2})
box_colors = ['#2D6A4F', '#40916C', '#52B788', '#F4A261']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
ax2.set_xticks(range(1, 5))
ax2.set_xticklabels(ship_order, rotation=12, ha='right', fontsize=8)
ax2.set_title('Çatdırılma Müddəti (Gün)', fontweight='bold')
ax2.set_ylabel('Gün')

# 3. Satış paylaşımı (histogram)
ax3 = axes[2]
ax3.hist(df['Sales'].clip(upper=2000), bins=50,
         color='#40916C', edgecolor='white', linewidth=0.5)
ax3.axvline(df['Sales'].median(), color='#E76F51', linestyle='--',
            linewidth=2, label=f'Median: ${df["Sales"].median():.0f}')
ax3.axvline(df['Sales'].mean(), color='#F4A261', linestyle='-.',
            linewidth=2, label=f'Ortalama: ${df["Sales"].mean():.0f}')
ax3.set_title('Satış Dağılımı (max $2K)', fontweight='bold')
ax3.set_xlabel('Satış ($)')
ax3.set_ylabel('Sifariş Sayı')
ax3.legend(fontsize=9)

plt.tight_layout()
plt.savefig('04_customer_delivery.png', bbox_inches='tight', dpi=130)
plt.show()

print(f'💡 Insight: Sean Miller ən yüksək satışlı müştəri → VIP müştəri olaraq xüsusi proqrama daxil edilməlidir.')
print(f'💡 Insight: Satışların böyük hissəsi $0-$500 aralığındadır; sağa uzanan quyruq yüksək dəyərli sifarişlər var.')

## 7. Kateqoriya × Region × Zaman Dərinlik Analizi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🔍 Kateqoriya × Region Dərinlik Analizi', fontsize=14, fontweight='bold')

# 1. Kateqoriya x Region satış heatmap
ax1 = axes[0]
pivot_cr = df.pivot_table(values='Sales', index='Sub-Category', columns='Region', aggfunc='sum').fillna(0)
sns.heatmap(pivot_cr, ax=ax1, annot=True, fmt=',.0f',
            cmap='YlGn', linewidths=0.4, linecolor='white',
            annot_kws={'size': 7.5})
ax1.set_title('Alt-Kateqoriya × Region Satış ($)', fontweight='bold')
ax1.set_xlabel(''); ax1.set_ylabel('')

# 2. Kateqoriya üzrə illik growth
ax2 = axes[1]
cat_year = df.groupby(['Year', 'Category'])['Sales'].sum().unstack('Category')
for col, color in zip(cat_year.columns, ['#2D6A4F', '#F4A261', '#E76F51']):
    ax2.plot(cat_year.index.astype(str), cat_year[col],
             marker='o', linewidth=2.2, color=color, label=col, markersize=7)
    for x, y in zip(cat_year.index.astype(str), cat_year[col]):
        ax2.annotate(f'${y/1000:.0f}K', (x, y),
                     textcoords='offset points', xytext=(0, 8),
                     ha='center', fontsize=8, color=color)
ax2.set_title('Kateqoriya üzrə İllik Satış Trendi', fontweight='bold')
ax2.set_ylabel('Satış ($)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax2.legend(title='Kateqoriya', fontsize=9)
ax2.set_xlabel('İl')

plt.tight_layout()
plt.savefig('05_deep_analysis.png', bbox_inches='tight', dpi=130)
plt.show()

# İllik artım hesabı
for cat in cat_year.columns:
    first_val = cat_year[cat].iloc[0]
    last_val = cat_year[cat].iloc[-1]
    growth = (last_val - first_val) / first_val * 100
    print(f'📊 {cat}: {cat_year.index[0]}→{cat_year.index[-1]} artım = %{growth:.1f}')

## 8. ✅ Business Insights — İcmal

In [ ]:
# Rəqəmsal xülasə
total_sales     = df['Sales'].sum()
total_orders    = df['Order ID'].nunique()
total_customers = df['Customer ID'].nunique()
avg_order_val   = df.groupby('Order ID')['Sales'].sum().mean()
top_cat         = df.groupby('Category')['Sales'].sum().idxmax()
top_region      = df.groupby('Region')['Sales'].sum().idxmax()
top_segment     = df.groupby('Segment')['Sales'].sum().idxmax()
top_subcat      = df.groupby('Sub-Category')['Sales'].sum().idxmax()
avg_delivery    = df['Delivery Days'].mean()

print('='*58)
print('       📊  SUPERSTORE BUSİNESS İNSİGHT XÜLASƏSİ')
print('='*58)
print(f'  Ümumi Satış:          ${total_sales:>12,.0f}')
print(f'  Unikal Sifariş:       {total_orders:>12,}')
print(f'  Unikal Müştəri:       {total_customers:>12,}')
print(f'  Ort. Sifariş Dəyəri:  ${avg_order_val:>12,.2f}')
print('-'*58)
print(f'  Ən Yüksək Kateqoriya: {top_cat}')
print(f'  Ən Yüksək Region:     {top_region}')
print(f'  Ən Yüksək Seqment:    {top_segment}')
print(f'  Ən Yüksək Sub-Cat:    {top_subcat}')
print(f'  Ort. Çatdırılma:      {avg_delivery:.1f} gün')
print('='*58)

print('''
🔑 ƏSAS TƏKLİFLƏR:

1. 📦 Technology ən az sifarişlə ən çox gəlir gətirir.
   → Upselling strategiyası tətbiq edilməlidir.

2. 🌍 West & East regionları üstündür.
   → Central regionda kampaniyalar artırılmalıdır.

3. 👥 Consumer seqmenti liderdir, lakin Corporate
   ortalama sifarişdə daha gəlirlidir.
   → B2B satış kanalına investisiya tövsiyə olunur.

4. 🚚 Standard Class sifarişlərin >50%-ni tutur.
   → Sürətli çatdırılma endirimlə təklif edilsə
     seqment miqrasiyası mümkündür.

5. 📅 Q4 (oktyabr-dekabr) ən güclü dövrdür.
   → İl sonu kampaniyaları daha erkən başlamalıdır.

6. 🏆 Top 10 müştəri ümumi satışın əhəmiyyətli
   hissəsini oluşturur → VIP Loyallıq Proqramı.
''')